# End-to-End Machine Learning Regression: House Price Prediction
**Author:** Chiemela Joseph Nwosu  
**Version:** v2 Clean Portfolio Edition  
**Original Learning Source:** *Mastering ChatGPT and Google Colab for Machine Learning* (Moscato)  

---

## About This Notebook

This is a cleaned, portfolio-ready version of the original chapter-based learning notebook. It preserves the full regression workflow from the original while improving:
- Dataset loading from a local path (`data/raw/`)
- Clean train/test split **before** any preprocessing that learns from data
- Consistent feature naming throughout
- Saved visuals to the `visuals/` folder
- Clear markdown explanations for each section

The original notebook (`ML_Regression_Analysis_Model_github.ipynb`) is preserved unchanged.

---

## Project Goal

Predict house sale prices using structured housing features. Compare Linear Regression, Ridge Regression, and Lasso Regression using cross-validation and held-out test metrics.

## 1. Imports and Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Ensure visuals directory exists
os.makedirs('../visuals', exist_ok=True)

print('Libraries loaded successfully.')
print(f'pandas {pd.__version__} | numpy {np.__version__} | sklearn imported')

## 2. Load Dataset

The dataset is the King County, WA house sales dataset. It contains 21,613 records and 21 columns covering property characteristics, location, and sale price.

**Target variable:** `price` (continuous — sale price in USD)

In [ ]:
DATA_PATH = '../data/raw/House_Prices.csv'
TARGET_COLUMN = 'price'

df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Target column: {TARGET_COLUMN}')
df.head()

## 3. Exploratory Data Analysis (EDA)

Before modeling, we explore the dataset to understand distributions, relationships, and potential data quality issues.

In [ ]:
# Basic statistics
print('=== Basic Statistics ===')
print(df.describe().round(2))

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Data Types ===')
print(df.dtypes)

### 3.1 Price Distribution

The target variable `price` is right-skewed, with a mean (~$540K) higher than the median (~$450K). High-value outliers (up to $7.7M) are present.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[TARGET_COLUMN], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Count')

sns.boxplot(y=df[TARGET_COLUMN], ax=axes[1], color='steelblue')
axes[1].set_title('Price Boxplot (Outlier View)')
axes[1].set_ylabel('Price (USD)')

plt.suptitle('House Price Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/price_distribution.png')

### 3.2 Correlation Heatmap

We examine correlations between all numerical features and the target variable to identify the strongest predictors.

In [ ]:
# Drop non-numeric columns for correlation
numeric_df = df.drop(columns=['id', 'date'], errors='ignore')

corr_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    annot_kws={'size': 7}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/correlation_heatmap.png')

# Top correlations with price
print('\nTop features correlated with price:')
print(corr_matrix[TARGET_COLUMN].sort_values(ascending=False).round(3))

## 4. Data Preprocessing

### 4.1 Feature Engineering

Following the original notebook's approach, we drop columns that are identifiers or have low predictive value for this baseline model:
- `id` — unique identifier, not a predictor
- `date` — sale date (not used in this baseline)
- `yr_renovated` — mostly zeros; low signal in baseline model

We retain `lat`, `long`, and `zipcode` as location proxies.

In [ ]:
COLUMNS_TO_DROP = ['id', 'date', 'yr_renovated']

df_clean = df.drop(columns=COLUMNS_TO_DROP)

# Confirm target is present and separate
assert TARGET_COLUMN in df_clean.columns, f'{TARGET_COLUMN} not found in dataframe'

X = df_clean.drop(columns=[TARGET_COLUMN])  # Feature matrix — does NOT include target
y = df_clean[TARGET_COLUMN]                 # Target vector

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')
print(f'Features: {list(X.columns)}')

### 4.2 Train/Test Split

We split **before** scaling to prevent data leakage. The scaler will be fit only on training data and applied to both sets.

- **Train:** 70% of data
- **Test:** 30% of data
- `random_state=42` for reproducibility

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Training set:  {X_train.shape[0]} samples')
print(f'Test set:      {X_test.shape[0]} samples')

### 4.3 Feature Scaling

StandardScaler is fit on training data only, then applied to both train and test sets. This prevents information from the test set leaking into the scaling parameters.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

print('Scaling complete.')
print(f'Train mean (first feature, should be ~0): {X_train_scaled[:, 0].mean():.4f}')
print(f'Train std  (first feature, should be ~1): {X_train_scaled[:, 0].std():.4f}')

## 5. Model Training

Three regression models are trained and compared:

| Model | Regularization | Key Property |
|---|---|---|
| Linear Regression | None | Baseline; fully interpretable |
| Ridge Regression | L2 penalty | Shrinks all coefficients; handles multicollinearity |
| Lasso Regression | L1 penalty | Can zero out coefficients; implicit feature selection |

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Lasso Regression':  Lasso(alpha=1.0)
}

# Train all models
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f'{name}: trained')

## 6. Model Evaluation

### 6.1 Held-Out Test Set Metrics

We evaluate each model on the held-out test set using:
- **MAE** — Mean Absolute Error (average prediction error in dollars)
- **MSE** — Mean Squared Error (penalizes large errors more heavily)
- **RMSE** — Root Mean Squared Error (same units as price)
- **R²** — Proportion of variance explained (1.0 = perfect)

In [ ]:
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'y_pred': y_pred}

# Display summary table
summary = pd.DataFrame({
    name: {
        'MAE':  f"${v['MAE']:,.0f}",
        'RMSE': f"${v['RMSE']:,.0f}",
        'R²':   f"{v['R2']:.4f}"
    }
    for name, v in results.items()
}).T

print('=== Test Set Performance ===')
print(summary.to_string())

### 6.2 Cross-Validation (5-Fold)

Cross-validation provides a more reliable estimate of generalization performance than a single train/test split. We use 5-fold CV on the training set.

In [ ]:
scoring = ['neg_mean_absolute_error', 'neg_mean_squared_error', 'r2']
cv_results = {}

for name, model in models.items():
    cv = cross_validate(model, X_train_scaled, y_train, cv=5, scoring=scoring)
    cv_results[name] = cv

print('=== 5-Fold Cross-Validation Results (Training Set) ===')
print(f'{"Model":<22} {"CV MAE (mean)":>16} {"CV MAE (std)":>14} {"CV R² (mean)":>14} {"CV R² (std)":>12}')
print('-' * 82)
for name, cv in cv_results.items():
    mae_mean = -np.mean(cv['test_neg_mean_absolute_error'])
    mae_std  =  np.std(cv['test_neg_mean_absolute_error'])
    r2_mean  =  np.mean(cv['test_r2'])
    r2_std   =  np.std(cv['test_r2'])
    print(f'{name:<22} ${mae_mean:>14,.0f} ${mae_std:>12,.0f} {r2_mean:>14.4f} {r2_std:>12.4f}')

## 7. Model Comparison Visuals

### 7.1 Model Comparison Bar Chart

In [ ]:
model_names = list(results.keys())
mae_vals  = [results[m]['MAE']  for m in model_names]
rmse_vals = [results[m]['RMSE'] for m in model_names]
r2_vals   = [results[m]['R2']   for m in model_names]

x = np.arange(len(model_names))
width = 0.3

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(x, mae_vals, color=['steelblue', 'coral', 'mediumseagreen'])
axes[0].set_title('MAE by Model')
axes[0].set_ylabel('MAE (USD)')
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=15, ha='right')
for i, v in enumerate(mae_vals): axes[0].text(i, v + 500, f'${v:,.0f}', ha='center', fontsize=8)

axes[1].bar(x, rmse_vals, color=['steelblue', 'coral', 'mediumseagreen'])
axes[1].set_title('RMSE by Model')
axes[1].set_ylabel('RMSE (USD)')
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=15, ha='right')
for i, v in enumerate(rmse_vals): axes[1].text(i, v + 500, f'${v:,.0f}', ha='center', fontsize=8)

axes[2].bar(x, r2_vals, color=['steelblue', 'coral', 'mediumseagreen'])
axes[2].set_title('R² by Model')
axes[2].set_ylabel('R²')
axes[2].set_ylim(0, 1)
axes[2].set_xticks(x); axes[2].set_xticklabels(model_names, rotation=15, ha='right')
for i, v in enumerate(r2_vals): axes[2].text(i, v + 0.01, f'{v:.4f}', ha='center', fontsize=8)

plt.suptitle('Model Comparison: Test Set Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/model_comparison_metrics.png')

### 7.2 Actual vs. Predicted (Linear Regression)

In [ ]:
y_pred_lr = results['Linear Regression']['y_pred']

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lr, alpha=0.3, color='steelblue', edgecolors='none', s=15)
min_val = min(y_test.min(), y_pred_lr.min())
max_val = max(y_test.max(), y_pred_lr.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Actual vs. Predicted House Prices\n(Linear Regression — Test Set)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../visuals/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/actual_vs_predicted.png')

### 7.3 Residual Plot

In [ ]:
residuals = y_test.values - y_pred_lr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_lr, residuals, alpha=0.3, color='steelblue', s=15)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Price (USD)')
axes[0].set_ylabel('Residual (USD)')
axes[0].set_title('Residuals vs. Predicted')

sns.histplot(residuals, kde=True, ax=axes[1], color='steelblue')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (USD)')
axes[1].set_title('Residual Distribution')

plt.suptitle('Residual Analysis — Linear Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/residual_plot.png')

## 8. Model Interpretability — Coefficient Importance

Because features are standardized, each coefficient represents the change in predicted price for a **one standard deviation increase** in that feature. Larger absolute values indicate stronger influence.

In [ ]:
lr_model = models['Linear Regression']
feature_names = X.columns.tolist()

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('=== Linear Regression Coefficients (sorted by magnitude) ===')
print(coef_df.to_string(index=False))

# Plot
colors = ['steelblue' if c > 0 else 'coral' for c in coef_df['Coefficient']]

plt.figure(figsize=(10, 7))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value (standardized features)')
plt.title('Feature Coefficients — Linear Regression\n(Blue = positive effect, Red = negative effect)', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../visuals/coefficient_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/coefficient_importance.png')

## 9. Final Model Selection and Summary

**Selected model: Linear Regression**

All three models produced nearly identical performance across both the held-out test set and 5-fold cross-validation. This indicates that regularization (Ridge/Lasso) did not provide meaningful improvement at default settings, and the feature set was already well-behaved after standardization.

Linear Regression was selected because:
- Performance is comparable to Ridge and Lasso
- Coefficients are fully interpretable
- It serves as a transparent, explainable baseline

**Key metric interpretation:**
- An R² of ~0.70 means the model explains approximately 70% of the variance in house prices
- MAE of ~$127K means predictions are off by about $127,000 on average
- The residual plot shows heteroscedasticity at higher price ranges — a known limitation of linear models on skewed targets

In [ ]:
print('=== Final Model Summary ===')
print(f'Model:  Linear Regression')
print(f'MAE:    ${results["Linear Regression"]["MAE"]:,.2f}')
print(f'MSE:    ${results["Linear Regression"]["MSE"]:,.2f}')
print(f'RMSE:   ${results["Linear Regression"]["RMSE"]:,.2f}')
print(f'R²:     {results["Linear Regression"]["R2"]:.4f}')
print()
print('Visuals saved to ../visuals/')
import os
for f in sorted(os.listdir('../visuals')):
    print(f'  - {f}')